In [5]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 72
import numpy as np
import pandas as pd
import os

import torch

# set GPGPU device
Device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# n_gpu = torch.cuda.device_count()
# print(torch.cuda.get_device_name(0))

In [6]:
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, models, transforms

# models.list_models()

In [7]:
MODEL_PATH= 'EP_models/'
os.environ['TORCH_HOME'] = MODEL_PATH  # model cache


# Load a pretrained model, otherwise a blank model will be loaded
model1 = models.resnet18(weights='IMAGENET1K_V1')
M = model1.fc.in_features
print(M)

# binary classes or nn.Linear(M, n_output)
model1.fc = nn.Linear(M, 2)
model1 = model1.to(Device)

loss_func1 = nn.CrossEntropyLoss()
optimizer1 = optim.SGD(model1.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
opti_lr_scheduler1 = lr_scheduler.StepLR(optimizer1, step_size=7, gamma=0.1)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to EP_models/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 58.1MB/s]


512


In [9]:
ds_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

folder = 'EP_datasets/hymenoptera_data'
Phases = ('train', 'valid')  # names must match the data folder names
ds = {_:datasets.ImageFolder(os.path.join(folder, _), ds_transforms[_]) for _ in Phases}
Dataloaders = {_:torch.utils.data.DataLoader(ds[_], batch_size=8, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True) for _ in Phases}

N = {_: len(ds[_]) for _ in Phases}
labels = ds['train'].classes

In [10]:
def train_model(_model, _loss_func, _optimizer, _scheduler, epochs=25, info=False):
    import copy
    import sys

    model_best, acc_best = copy.deepcopy(_model.state_dict()), 0.0

    # Training loop
    for epoch in range(epochs):
        for phase in Phases:
            _model.train() if phase == 'train' else _model.eval()

            totloss = 0.0
            n_correct = 0

            for data in Dataloaders[phase]:
                X = data[0].to(Device, non_blocking=True)
                y = data[1].to(Device, non_blocking=True)

                _optimizer.zero_grad()
                with torch.set_grad_enabled(phase=='train'):
                    net_out  = _model(X)
                    _, preds = torch.max(net_out, 1)
                    loss = _loss_func(net_out, y)

                    if phase == 'train':
                        loss.backward()  # compute gradients
                        _optimizer.step()

                totloss += loss.item()*X.size(0)
                n_correct += torch.sum(preds==y.data)
            if phase == 'train':
                _scheduler.step()

            acc_epoch = n_correct.double()/N[phase]
            if info:
                sys.stderr.write(f"\r{epoch+1:03d}/{epochs:3d} | {phase} | Loss: {totloss/N[phase]:6.2f} | "
                                 f"Avg loss: {totloss/(epoch+1):6.2f} | Acc: {acc_epoch:.3f}")
                sys.stderr.flush()

            # deep copy the model
            if phase == 'valid' and acc_epoch > acc_best:
                model_best, acc_best = copy.deepcopy(_model.state_dict()), acc_epoch

    print(f'Best validation Acc= {acc_best:.3f}')

    # load best model weights
    _model.load_state_dict(model_best)
    return _model

In [11]:
%%time

model1 = train_model(model1, loss_func1, optimizer1, opti_lr_scheduler1, epochs=100, info=True)

/Users/jeff/.local/share/mamba/envs/jhu-genai-mod6/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100/100 | valid | Loss:   0.15 | Avg loss:   0.23 | Acc: 0.948

Best validation Acc= 0.954
CPU times: user 2h 3min 37s, sys: 18min 7s, total: 2h 21min 44s
Wall time: 25min 51s
